# Athena emulator parity — boto3 query lifecycle

Third notebook of the `athena_emulator` parity series: exercises the boto3 query-plane lifecycle against the live stack — start→poll→results, `GetQueryResults` pagination, `StopQueryExecution`, batch/list executions, runtime statistics, DDL via `StartQueryExecution` (`CREATE DATABASE` dialect map, CTAS manifest), error paths, and the submit-time option members (`ClientRequestToken`, `ResultReuseConfiguration`, `ExecutionParameters`, `EncryptionConfiguration`, non-default `Catalog`). Each probe records a measured `PASS`/`FAIL`/`GAP`; the session matrix persists to `PARITY.md`.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) · `trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).

In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid
from collections.abc import Callable
from dataclasses import dataclass

import boto3
from _evidence import persist, record, render
from botocore.exceptions import ClientError

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL

RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb03- prefix (nb03_ for Glue
# databases, where '-' is invalid) so cleanup can discover leftovers — its
# own and any failed run's — instead of tracking them in a list.
ATHENA_RESOURCE_PREFIX = "nb03-"
DATABASE_PREFIX = "nb03_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
workgroup = f"{ATHENA_RESOURCE_PREFIX}wg-{RUN_SUFFIX}"
ddl_database = f"{DATABASE_PREFIX}ddl_{RUN_SUFFIX}"
ctas_table = f"{DATABASE_PREFIX}ctas_{RUN_SUFFIX}"
# Model pattern [a-zA-Z_][a-zA-Z0-9_@:]{1,256} — no dash, no leading digit.
statement_name = f"nb03_stmt_{RUN_SUFFIX}"
# Trino caps sequence() at 10000 entries (measured in the CLI consumer
# suite), so three capped unnests cross-joined stay RUNNING long enough
# for a client round-trip to land a GetQueryResults read and a stop.
SLOW_QUERY = (
    "SELECT count(*) FROM unnest(sequence(1, 10000)) a "
    "CROSS JOIN unnest(sequence(1, 10000)) b "
    "CROSS JOIN unnest(sequence(1, 10000)) c"
)

s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})
# Enforced ResultConfiguration: every probe's artifacts land under
# results/ without repeating OutputLocation per StartQueryExecution call.
athena.create_work_group(
    Name=workgroup,
    Configuration={
        "EnforceWorkGroupConfiguration": True,
        "ResultConfiguration": {"OutputLocation": f"s3://{bucket}/results/"},
    },
)

{'ResponseMetadata': {'RequestId': 'd4273cab-872f-4175-8105-6e8b45386769',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 06 Oct 2026 21:58:12 GMT',
   'server': 'uvicorn',
   'content-type': 'application/x-amz-json-1.1',
   'x-amzn-requestid': 'd4273cab-872f-4175-8105-6e8b45386769',
   'content-length': '2'},
  'RetryAttempts': 0}}

## Helpers

Shared plumbing for the probes below: capture `ClientError` shapes, start queries in the run workgroup (with `StartQueryExecution` request members as keyword overrides), poll to a terminal state, read moto-S3 objects, and sweep every `nb03`-prefixed resource at the end.

In [3]:
@dataclass(frozen=True)
class QueryRun:
    """Terminal outcome of one StartQueryExecution → poll cycle."""

    query_id: str
    state: str
    latency_ms: float
    execution: dict[str, object]


def client_error_of(call: Callable[[], object]) -> ClientError | None:
    """The ClientError ``call`` raised, or None when it succeeded."""
    try:
        call()
    except ClientError as exc:
        return exc
    return None


def error_code_status(error: ClientError) -> tuple[str, int]:
    """(Error.Code, HTTPStatusCode) of a captured ClientError."""
    return (
        error.response["Error"]["Code"],
        error.response["ResponseMetadata"]["HTTPStatusCode"],
    )


def query_execution(query_id: str) -> dict[str, object]:
    """The GetQueryExecution ``QueryExecution`` member for ``query_id``."""
    return athena.get_query_execution(QueryExecutionId=query_id)[
        "QueryExecution"
    ]


def start_query(query: str, **overrides: object) -> str:
    """Start ``query`` in the run workgroup; return its QueryExecutionId."""
    request: dict[str, object] = {
        "QueryString": query,
        "QueryExecutionContext": {"Database": database},
        "WorkGroup": workgroup,
    }
    request.update(overrides)
    return athena.start_query_execution(**request)["QueryExecutionId"]


def wait_terminal(query_id: str, timeout_s: float = 90.0) -> dict[str, object]:
    """Poll GetQueryExecution until ``query_id`` leaves QUEUED/RUNNING."""
    deadline = time.monotonic() + timeout_s
    execution = query_execution(query_id)
    while execution["Status"]["State"] in {"QUEUED", "RUNNING"}:
        if time.monotonic() > deadline:
            raise TimeoutError(
                f"{query_id} still {execution['Status']['State']} "
                f"after {timeout_s}s"
            )
        time.sleep(0.25)
        execution = query_execution(query_id)
    return execution


def run_to_terminal(query: str, **overrides: object) -> QueryRun:
    """Start ``query`` and poll it to a terminal state."""
    started = time.monotonic()
    query_id = start_query(query, **overrides)
    execution = wait_terminal(query_id)
    return QueryRun(
        query_id=query_id,
        state=execution["Status"]["State"],
        latency_ms=(time.monotonic() - started) * 1000,
        execution=execution,
    )


def object_keys_under(prefix: str) -> list[str]:
    """Object keys under ``prefix`` in the throwaway bucket."""
    contents = s3.list_objects_v2(Bucket=bucket, Prefix=prefix).get(
        "Contents", []
    )
    return [obj["Key"] for obj in contents]


def read_object(key: str) -> bytes:
    """One moto-S3 object's bytes from the throwaway bucket."""
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read()


def run_workgroup_names() -> list[str]:
    """Workgroup names carrying this run's (or a failed run's) prefix."""
    return [
        summary["Name"]
        for summary in athena.list_work_groups()["WorkGroups"]
        if summary["Name"].startswith(ATHENA_RESOURCE_PREFIX)
    ]


def delete_workgroup_contents(workgroup_name: str) -> None:
    """Delete the named queries and prepared statements in a workgroup."""
    queries = athena.list_named_queries(WorkGroup=workgroup_name)
    for query_id in queries.get("NamedQueryIds", []):
        athena.delete_named_query(NamedQueryId=query_id)
    statements = athena.list_prepared_statements(WorkGroup=workgroup_name)
    for summary in statements.get("PreparedStatements", []):
        athena.delete_prepared_statement(
            StatementName=summary["StatementName"], WorkGroup=workgroup_name
        )


def delete_run_buckets() -> None:
    """Empty and delete every bucket carrying the run prefix."""
    for bucket_info in s3.list_buckets()["Buckets"]:
        name = bucket_info["Name"]
        if not name.startswith(ATHENA_RESOURCE_PREFIX):
            continue
        objects = s3.list_objects_v2(Bucket=name).get("Contents", [])
        if objects:
            s3.delete_objects(
                Bucket=name,
                Delete={"Objects": [{"Key": obj["Key"]} for obj in objects]},
            )
        s3.delete_bucket(Bucket=name)


def delete_run_databases() -> None:
    """Delete tables, then every Glue database carrying the run prefix."""
    for database_info in glue.get_databases()["DatabaseList"]:
        database_name = database_info["Name"]
        if not database_name.startswith(DATABASE_PREFIX):
            continue
        tables = glue.get_tables(DatabaseName=database_name)["TableList"]
        for table in tables:
            glue.delete_table(DatabaseName=database_name, Name=table["Name"])
        glue.delete_database(Name=database_name)


def cleanup_run_resources() -> None:
    """Remove everything this run (or a failed earlier run) left behind."""
    for name in run_workgroup_names():
        delete_workgroup_contents(name)
        athena.delete_work_group(WorkGroup=name)
    delete_run_buckets()
    delete_run_databases()

## Lifecycle — start → poll → results

The contract a poller sees: `StartQueryExecution` returns the ID without waiting, `GetQueryExecution` reports `QUEUED`/`RUNNING` then a terminal state with `CompletionDateTime`, `StatementType`/`SubstatementType` are classified at submit, and a DML `OutputLocation` names the `{QueryID}.csv` artifact itself.

In [4]:
select_run = run_to_terminal(
    "SELECT * FROM (VALUES (1,'a'),(2,'b'),(3,'c'),(4,'d'),(5,'e')) t(n,s)"
)
assert select_run.state == "SUCCEEDED"
assert select_run.execution["StatementType"] == "DML"
assert select_run.execution["SubstatementType"] == "SELECT"
assert select_run.execution["QueryExecutionContext"]["Database"] == database
output_location = select_run.execution["ResultConfiguration"]["OutputLocation"]
assert output_location.endswith(f"{select_run.query_id}.csv")
assert f"results/{select_run.query_id}.csv" in object_keys_under("results/")

record(
    "start→poll→GetQueryExecution (SELECT) + .csv artifact",
    "PASS",
    f"{select_run.query_id} {select_run.state} DML/SELECT; "
    f"OutputLocation={output_location}",
    select_run.latency_ms,
)

Evidence(feature='start→poll→GetQueryExecution (SELECT) + .csv artifact', status='PASS', detail='92d55cc1-c8fb-4bb0-b431-db3bf4cddb82 SUCCEEDED DML/SELECT; OutputLocation=s3://nb03-3216e3a14ca8/results/92d55cc1-c8fb-4bb0-b431-db3bf4cddb82.csv', latency_ms=82.02700401307084)

In [5]:
# The wire contract: the header row travels on page zero only, NextToken
# is an opaque continuation, and botocore's paginator merges the pages
# losslessly (the path wrangler's _fetch_api_result walks).
first_page = athena.get_query_results(
    QueryExecutionId=select_run.query_id, MaxResults=2
)
first_rows = first_page["ResultSet"]["Rows"]
assert [c["VarCharValue"] for c in first_rows[0]["Data"]] == ["n", "s"]
assert len(first_rows) == 3  # header + two data rows
assert "NextToken" in first_page

middle_page = athena.get_query_results(
    QueryExecutionId=select_run.query_id,
    MaxResults=2,
    NextToken=first_page["NextToken"],
)
middle_rows = middle_page["ResultSet"]["Rows"]
assert [c["VarCharValue"] for c in middle_rows[0]["Data"]] == ["3", "c"]
assert len(middle_rows) == 2
assert "NextToken" in middle_page

last_page = athena.get_query_results(
    QueryExecutionId=select_run.query_id,
    MaxResults=2,
    NextToken=middle_page["NextToken"],
)
assert len(last_page["ResultSet"]["Rows"]) == 1
assert "NextToken" not in last_page

paged_rows = []
paginator = athena.get_paginator("get_query_results")
for page in paginator.paginate(
    QueryExecutionId=select_run.query_id,
    PaginationConfig={"PageSize": 2},
):
    paged_rows.extend(page["ResultSet"]["Rows"])
assert len(paged_rows) == 6  # header + five data rows
assert [
    c["Name"]
    for c in first_page["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]
] == ["n", "s"]

record(
    "GetQueryResults paginates (header on page 0, opaque NextToken)",
    "PASS",
    "3 manual pages (3+2+1 rows); botocore paginator merged 6 rows losslessly",
)

Evidence(feature='GetQueryResults paginates (header on page 0, opaque NextToken)', status='PASS', detail='3 manual pages (3+2+1 rows); botocore paginator merged 6 rows losslessly', latency_ms=None)

In [6]:
# A second execution gives the list probe two run-scoped rows.
second_run = run_to_terminal("SELECT 2 AS two")
assert second_run.state == "SUCCEEDED"

batch = athena.batch_get_query_execution(
    QueryExecutionIds=[
        select_run.query_id,
        "00000000-0000-0000-0000-000000000000",
    ]
)
assert {e["QueryExecutionId"] for e in batch["QueryExecutions"]} == {
    select_run.query_id
}
unprocessed = batch["UnprocessedQueryExecutionIds"]
assert [u["QueryExecutionId"] for u in unprocessed] == [
    "00000000-0000-0000-0000-000000000000"
]
assert unprocessed[0]["ErrorCode"] == "InvalidRequestException"

# Most-recent-first, MaxResults-bounded pages — the path wrangler's cache
# probe walks before batch_getting each ID.
seen_ids = []
token = None
while True:
    page_args: dict[str, object] = {
        "WorkGroup": workgroup,
        "MaxResults": 1,
    }
    if token is not None:
        page_args["NextToken"] = token
    page = athena.list_query_executions(**page_args)
    seen_ids.extend(page["QueryExecutionIds"])
    token = page.get("NextToken")
    if token is None:
        break
assert seen_ids == [second_run.query_id, select_run.query_id]

record(
    "batch_get/list executions (unprocessed IDs, paged, newest first)",
    "PASS",
    "unknown id → UnprocessedQueryExecutionIds; MaxResults=1 paginated "
    f"{len(seen_ids)} executions newest-first",
)

Evidence(feature='batch_get/list executions (unprocessed IDs, paged, newest first)', status='PASS', detail='unknown id → UnprocessedQueryExecutionIds; MaxResults=1 paginated 2 executions newest-first', latency_ms=None)

In [7]:
statistics = athena.get_query_runtime_statistics(
    QueryExecutionId=select_run.query_id
)["QueryRuntimeStatistics"]
engine_ms = statistics["Timeline"]["EngineExecutionTimeInMillis"]
input_bytes = statistics["Rows"]["InputBytes"]
assert isinstance(engine_ms, int)
assert isinstance(input_bytes, int)

record(
    "GetQueryRuntimeStatistics (Timeline + Rows shape)",
    "PASS",
    f"EngineExecutionTimeInMillis={engine_ms} InputBytes={input_bytes}",
)

Evidence(feature='GetQueryRuntimeStatistics (Timeline + Rows shape)', status='PASS', detail='EngineExecutionTimeInMillis=0 InputBytes=0', latency_ms=None)

## Stop and pre-finish reads

A `GetQueryResults` call against a non-terminal execution answers `InvalidRequestException` 400 naming the current state — the rejection a premature consumer sees. `StopQueryExecution` cancels the in-flight Trino statement and lands the execution on `CANCELLED`. Real AWS treats stop on an already-terminal execution as a 200 no-op; whatever the emulator does is measured.

In [8]:
slow_id = start_query(SLOW_QUERY)
feature = "GetQueryResults on a running execution → 400"
pre_finish_error = client_error_of(
    lambda: athena.get_query_results(QueryExecutionId=slow_id)
)
if pre_finish_error is None:
    record(feature, "FAIL", "returned rows for a non-terminal execution")
else:
    code, status = error_code_status(pre_finish_error)
    record(
        feature,
        "PASS"
        if (code, status) == ("InvalidRequestException", 400)
        else "FAIL",
        f"{code} ({status}): {pre_finish_error.response['Error']['Message']}",
    )

athena.stop_query_execution(QueryExecutionId=slow_id)
stopped = wait_terminal(slow_id)
record(
    "StopQueryExecution → CANCELLED",
    "PASS" if stopped["Status"]["State"] == "CANCELLED" else "FAIL",
    f"{slow_id} {stopped['Status']['State']}",
)

feature = "stop on a terminal execution (AWS no-ops with 200)"
terminal_stop_error = client_error_of(
    lambda: athena.stop_query_execution(QueryExecutionId=select_run.query_id)
)
if terminal_stop_error is None:
    record(feature, "PASS", "200 no-op on a SUCCEEDED execution")
else:
    code, status = error_code_status(terminal_stop_error)
    record(
        feature,
        "FAIL",
        f"{code} ({status}) on a SUCCEEDED execution; "
        "real AWS answers a 200 no-op",
    )

## DDL via `StartQueryExecution`

`CREATE DATABASE` is Athena vocabulary Trino lacks — the emulator dialect-maps it to `CREATE SCHEMA` before submit (backtick and `IF NOT EXISTS` forms covered), so the Glue metastore behind Trino gains a real database. DDL results are the headerless tab `{QueryID}.txt` plus `.txt.metadata`. A CTAS is also `DDL` on the wire (`CREATE_TABLE_AS_SELECT`) but reports the bare `OutputLocation` stem and names its `{QueryID}-manifest.csv` via `Statistics.DataManifestLocation` — the file list a consumer (wrangler's CTAS path) reads to find the parquet files the query wrote at `external_location`.

In [9]:
create_run = run_to_terminal(f"CREATE DATABASE {ddl_database}")
assert create_run.state == "SUCCEEDED"
assert create_run.execution["StatementType"] == "DDL"
assert create_run.execution["SubstatementType"] == "CREATE_DATABASE"
# The stored Query is the text as submitted, not the mapped dialect.
assert create_run.execution["Query"] == f"CREATE DATABASE {ddl_database}"
ddl_output = create_run.execution["ResultConfiguration"]["OutputLocation"]
assert ddl_output.endswith(f"{create_run.query_id}.txt")

# The Glue metastore behind Trino (moto) must hold the new database.
assert glue.get_database(Name=ddl_database)["Database"]["Name"] == ddl_database
assert sorted(
    k
    for k in object_keys_under("results/")
    if k.startswith(f"results/{create_run.query_id}")
) == [
    f"results/{create_run.query_id}.txt",
    f"results/{create_run.query_id}.txt.metadata",
]

record(
    "CREATE DATABASE → Glue database + .txt artifact (dialect map)",
    "PASS",
    f"{create_run.query_id} {create_run.state} DDL/CREATE_DATABASE; "
    f"{ddl_database} visible via glue.get_database; "
    ".txt + .txt.metadata written",
    create_run.latency_ms,
)

Evidence(feature='CREATE DATABASE → Glue database + .txt artifact (dialect map)', status='PASS', detail='83eea215-f53f-497c-af1c-542c63cbcce5 SUCCEEDED DDL/CREATE_DATABASE; nb03_ddl_4815a2e6 visible via glue.get_database; .txt + .txt.metadata written', latency_ms=295.0448560004588)

In [10]:
ctas_run = run_to_terminal(
    f"CREATE TABLE {ctas_table} "
    f"WITH (format='PARQUET', "
    f"external_location='s3://{bucket}/ctas-data/') "
    f"AS SELECT 1 AS n, 'x' AS s"
)
assert ctas_run.state == "SUCCEEDED"
assert ctas_run.execution["StatementType"] == "DDL"
assert ctas_run.execution["SubstatementType"] == "CREATE_TABLE_AS_SELECT"

# Manifest statements report the bare stem; Statistics names the manifest.
ctas_stem = f"s3://{bucket}/results/{ctas_run.query_id}"
assert ctas_run.execution["ResultConfiguration"]["OutputLocation"] == ctas_stem
manifest_location = ctas_run.execution["Statistics"]["DataManifestLocation"]
assert manifest_location == f"{ctas_stem}-manifest.csv"
manifest_key = manifest_location.removeprefix(f"s3://{bucket}/")
manifest_entries = read_object(manifest_key).decode().splitlines()
assert manifest_entries
assert all(
    entry.startswith(f"s3://{bucket}/ctas-data/") for entry in manifest_entries
)
assert f"results/{ctas_run.query_id}.metadata" in object_keys_under("results/")

# The Trino hive connector registered the table in the Glue catalog.
registered = glue.get_table(DatabaseName=database, Name=ctas_table)["Table"]
assert registered["StorageDescriptor"]["Location"] == (
    f"s3://{bucket}/ctas-data/"
)

record(
    "CTAS → -manifest.csv + .metadata + Glue table (parquet at "
    "external_location)",
    "PASS",
    f"{ctas_run.query_id} {ctas_run.state} DDL/CREATE_TABLE_AS_SELECT; "
    f"manifest lists {len(manifest_entries)} file(s) under ctas-data/",
    ctas_run.latency_ms,
)

Evidence(feature='CTAS → -manifest.csv + .metadata + Glue table (parquet at external_location)', status='PASS', detail='8f0c9161-dba7-4c92-8946-aac50af4b277 SUCCEEDED DDL/CREATE_TABLE_AS_SELECT; manifest lists 1 file(s) under ctas-data/', latency_ms=313.4766619914444)

## Error paths

Athena splits failures by surface: a **syntax** failure is rejected at submit with a 400 `InvalidRequestException` (`"Exception parsing query…"`, the prefix wrangler sniffs), while an **analysis** failure (missing table) still returns an ID and lands the execution on `FAILED` with the engine's reason in `StateChangeReason`. Unknown execution IDs are `InvalidRequestException` 400, and `primary` — which carries no `OutputLocation` — rejects a submission without `ResultConfiguration`.

In [11]:
syntax_error = client_error_of(lambda: start_query("SELEC T oops"))
assert syntax_error is not None
code, status = error_code_status(syntax_error)
assert (code, status) == ("InvalidRequestException", 400)
assert syntax_error.response["Error"]["Message"].startswith(
    "Exception parsing query"
)
record(
    "syntax error → submit-time InvalidRequestException 400",
    "PASS",
    f"{code} ({status}); message starts 'Exception parsing query'",
)

missing_table_run = run_to_terminal("SELECT * FROM no_such_table_xyz")
assert missing_table_run.state == "FAILED"
reason = missing_table_run.execution["Status"]["StateChangeReason"]
assert reason
record(
    "analysis error → FAILED execution with StateChangeReason",
    "PASS",
    f"{missing_table_run.query_id} FAILED: {reason.splitlines()[0][:90]}",
)

missing_execution_error = client_error_of(
    lambda: query_execution("00000000-0000-0000-0000-000000000000")
)
assert missing_execution_error is not None
code, status = error_code_status(missing_execution_error)
assert (code, status) == ("InvalidRequestException", 400)
record(
    "unknown QueryExecutionId → InvalidRequestException 400",
    "PASS",
    f"{code} ({status})",
)

missing_output_error = client_error_of(
    lambda: athena.start_query_execution(
        QueryString="SELECT 1", WorkGroup="primary"
    )
)
assert missing_output_error is not None
code, status = error_code_status(missing_output_error)
assert (code, status) == ("InvalidRequestException", 400)
record(
    "missing OutputLocation (primary, no ResultConfiguration) → 400",
    "PASS",
    f"{code} ({status})",
)

Evidence(feature='missing OutputLocation (primary, no ResultConfiguration) → 400', status='PASS', detail='InvalidRequestException (400)', latency_ms=None)

## Prepared-statement execution and `ExecutionParameters`

`EXECUTE <name>` resolves the stored statement server-side and binds `ExecutionParameters` (or inline `USING` values) into its `?` markers — the member is echoed back on `GetQueryExecution`. A bare `?` on a plain query takes the same binding (wrangler's `paramstyle="qmark"` path: bare values coerce by context type, so `"42"` binds numeric). A missing statement or a `?`-count mismatch fails the execution with Athena's `StateChangeReason` phrasing — never a 400.

In [12]:
athena.create_prepared_statement(
    StatementName=statement_name,
    WorkGroup=workgroup,
    QueryStatement="SELECT ? AS chosen",
)

exec_run = run_to_terminal(
    f"EXECUTE {statement_name}", ExecutionParameters=["hello"]
)
assert exec_run.state == "SUCCEEDED"
assert exec_run.execution["ExecutionParameters"] == ["hello"]
exec_rows = athena.get_query_results(QueryExecutionId=exec_run.query_id)[
    "ResultSet"
]["Rows"]
assert exec_rows[1]["Data"][0]["VarCharValue"] == "hello"
record(
    "EXECUTE + ExecutionParameters binds and echoes",
    "PASS",
    f"{statement_name} → row 'hello'; ExecutionParameters echoed",
    exec_run.latency_ms,
)

qmark_run = run_to_terminal("SELECT ? AS chosen", ExecutionParameters=["42"])
assert qmark_run.state == "SUCCEEDED"
qmark_rows = athena.get_query_results(QueryExecutionId=qmark_run.query_id)[
    "ResultSet"
]["Rows"]
assert qmark_rows[1]["Data"][0]["VarCharValue"] == "42"
record(
    "qmark ExecutionParameters on a plain query",
    "PASS",
    "SELECT ? with ['42'] → row '42'",
    qmark_run.latency_ms,
)

missing_stmt_run = run_to_terminal(f"EXECUTE {statement_name}_missing")
assert missing_stmt_run.state == "FAILED"
assert (
    "was not found in workGroup"
    in (missing_stmt_run.execution["Status"]["StateChangeReason"])
)
mismatch_run = run_to_terminal(
    f"EXECUTE {statement_name}", ExecutionParameters=["a", "b"]
)
assert mismatch_run.state == "FAILED"
assert (
    "Incorrect number of parameters"
    in (mismatch_run.execution["Status"]["StateChangeReason"])
)
record(
    "EXECUTE failures → FAILED not 400 (missing stmt, ?-count)",
    "PASS",
    "both FAILED with Athena's StateChangeReason phrasing",
)

Evidence(feature='EXECUTE failures → FAILED not 400 (missing stmt, ?-count)', status='PASS', detail="both FAILED with Athena's StateChangeReason phrasing", latency_ms=None)

## Submit-time option members

The model declares four `StartQueryExecution` members the lifecycle doesn't depend on but consumers send: `ClientRequestToken` (AWS dedupes a retried submission — same token, same `QueryExecutionId`), `ResultReuseConfiguration` (AWS re-answers a recent identical query and reports `Statistics.ResultReuseInformation.ReusedPreviousResult`), `ResultConfiguration.EncryptionConfiguration` (echoed verbatim on `GetQueryExecution`), and `QueryExecutionContext.Catalog` (a registered catalog name — `AwsDataCatalog` is the seeded GLUE one; AWS rejects unknown catalogs). Each is measured, never assumed.

In [13]:
feature = "ClientRequestToken dedupes retried submissions"
token = str(uuid.uuid4())
token_first = start_query("SELECT 1 AS one", ClientRequestToken=token)
token_second = start_query("SELECT 1 AS one", ClientRequestToken=token)
if token_first == token_second:
    record(
        feature,
        "PASS",
        f"token {token[:8]}… → single execution {token_first[:8]}…",
    )
else:
    record(
        feature,
        "GAP",
        f"token {token[:8]}… submitted twice → distinct executions "
        f"{token_first[:8]}…/{token_second[:8]}…; "
        "AWS returns the original for a retried token",
    )

In [14]:
feature = "ResultReuseConfiguration reuses recent results"
reuse_configuration = {
    "ResultReuseByAgeConfiguration": {
        "Enabled": True,
        "MaxAgeInMinutes": 60,
    }
}
reuse_first = run_to_terminal(
    "SELECT 7 AS seven", ResultReuseConfiguration=reuse_configuration
)
reuse_second = run_to_terminal(
    "SELECT 7 AS seven", ResultReuseConfiguration=reuse_configuration
)
reused = (
    reuse_second.execution.get("Statistics", {})
    .get("ResultReuseInformation", {})
    .get("ReusedPreviousResult")
)
echoed = "ResultReuseConfiguration" in reuse_second.execution
if reused and echoed:
    record(
        feature,
        "PASS",
        f"second identical run reused (ReusedPreviousResult={reused})",
    )
else:
    record(
        feature,
        "GAP",
        f"second identical run {reuse_second.state} "
        f"(ReusedPreviousResult={reused}, config echoed={echoed}); "
        "AWS re-answers the recent result",
    )

In [15]:
encryption = {
    "EncryptionOption": "SSE_S3",
    "KmsKey": f"arn:aws:kms:us-east-1:123456789012:key/nb03-{RUN_SUFFIX}",
}
enc_run = run_to_terminal(
    "SELECT 1",
    ResultConfiguration={
        "OutputLocation": f"s3://{bucket}/enc/",
        "EncryptionConfiguration": encryption,
    },
)
assert enc_run.state == "SUCCEEDED"
reported = enc_run.execution["ResultConfiguration"]
# The workgroup is enforced, so its OutputLocation still wins; the
# request's EncryptionConfiguration is the member under test.
assert reported["EncryptionConfiguration"] == encryption
record(
    "ResultConfiguration.EncryptionConfiguration echoed verbatim",
    "PASS",
    "SSE_S3 + KmsKey echoed on GetQueryExecution",
    enc_run.latency_ms,
)

Evidence(feature='ResultConfiguration.EncryptionConfiguration echoed verbatim', status='PASS', detail='SSE_S3 + KmsKey echoed on GetQueryExecution', latency_ms=317.59468400559854)

In [16]:
catalog_run = run_to_terminal(
    "SELECT 1",
    QueryExecutionContext={
        "Database": database,
        "Catalog": "AwsDataCatalog",
    },
)
assert catalog_run.state == "SUCCEEDED"
assert (
    catalog_run.execution["QueryExecutionContext"]["Catalog"]
    == "AwsDataCatalog"
)
record(
    "QueryExecutionContext.Catalog=AwsDataCatalog round-trips",
    "PASS",
    f"{catalog_run.query_id} {catalog_run.state}; catalog echoed",
    catalog_run.latency_ms,
)

feature = "unknown catalog rejected at submit"
bogus_catalog = f"{DATABASE_PREFIX}nocat_{RUN_SUFFIX}"
try:
    bogus_id = start_query(
        "SELECT 1", QueryExecutionContext={"Catalog": bogus_catalog}
    )
    bogus_state = wait_terminal(bogus_id)["Status"]["State"]
    record(
        feature,
        "GAP",
        f"{bogus_catalog} accepted → {bogus_state} on the hive catalog "
        "regardless; real AWS rejects unregistered catalogs",
    )
except ClientError as exc:
    code, status = error_code_status(exc)
    record(feature, "PASS", f"{code} ({status})")

## Session matrix

Persisted to `parity/03_boto3_query_lifecycle.md`; `PARITY.md` at the project root is regenerated from all fragments.

In [17]:
print(render())
persist("03_boto3_query_lifecycle")

| feature | status | detail | latency_ms |
|---|---|---|---|
| start→poll→GetQueryExecution (SELECT) + .csv artifact | PASS | 92d55cc1-c8fb-4bb0-b431-db3bf4cddb82 SUCCEEDED DML/SELECT; OutputLocation=s3://nb03-3216e3a14ca8/results/92d55cc1-c8fb-4bb0-b431-db3bf4cddb82.csv | 82.027 |
| GetQueryResults paginates (header on page 0, opaque NextToken) | PASS | 3 manual pages (3+2+1 rows); botocore paginator merged 6 rows losslessly |  |
| batch_get/list executions (unprocessed IDs, paged, newest first) | PASS | unknown id → UnprocessedQueryExecutionIds; MaxResults=1 paginated 2 executions newest-first |  |
| GetQueryRuntimeStatistics (Timeline + Rows shape) | PASS | EngineExecutionTimeInMillis=0 InputBytes=0 |  |
| GetQueryResults on a running execution → 400 | PASS | InvalidRequestException (400): Query has not yet finished. Current state: RUNNING |  |
| StopQueryExecution → CANCELLED | PASS | 81e8ab9e-352e-4932-a37b-d9f964af6240 CANCELLED |  |
| stop on a terminal execution (AWS no-ops wit

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/03_boto3_query_lifecycle.md')

In [18]:
cleanup_run_resources()
print(
    "cleaned: "
    f"workgroups={run_workgroup_names()} "
    "buckets+glue databases (incl. tables) with the nb03 prefix removed"
)

cleaned: workgroups=[] buckets+glue databases (incl. tables) with the nb03 prefix removed
